In [ ]:
# ============================================================
# 1. Konfigurasi
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import re, os

SAVE_DIR  = r"C:\Users\Asus\skripsi"
THRESHOLD = 1  # skor minimum untuk label distorsi

path_gabung   = os.path.join(SAVE_DIR, "dataset_gabungan.csv")
path_labeled  = os.path.join(SAVE_DIR, "dataset_labeled.csv")
path_balanced = os.path.join(SAVE_DIR, "dataset_balanced.csv")

print(f"📁 Direktori : {SAVE_DIR}")
print(f"🔢 Threshold : {THRESHOLD}")


# ============================================================
# 2. Labeling
# ============================================================

KATA_DISTORSI = [
    r'\bgeger\b', r'\bheboh\b', r'\bgempar\b', r'\bmenggemparkan\b',
    r'\bmenghebohkan\b', r'\bbikin heboh\b', r'\bbikin gempar\b', r'\bbikin geger\b',
    r'\bbikin merinding\b', r'\bbikin kaget\b', r'\bbikin panik\b',
    r'\bnetizen heboh\b', r'\bnetizen geram\b', r'\bnetizen murka\b',
    r'\bramai diperbincangkan\b', r'\bramai dibicarakan\b',
    r'\bbikin geleng-geleng\b', r'\bpanen komentar\b', r'\bbanjir komentar\b',
    r'\bwajib tahu\b', r'\bfakta mengejutkan\b', r'\bfakta tersembunyi\b',
    r'\bfakta mencengangkan\b', r'\bbikin penasaran\b', r'\bmencuri perhatian\b',
    r'\bviral\b', r'\bsyok\b', r'\bshocking\b',
    r'\bseluruh rakyat\b', r'\bseluruh Indonesia\b', r'\bseluruh dunia\b',
    r'\bsepanjang sejarah\b', r'\bsepanjang masa\b',
    r'\bpertama kali dalam sejarah\b', r'\bbelum pernah terjadi\b',
    r'\bterbesar sepanjang masa\b', r'\bterparah sepanjang masa\b',
    r'\bkonspirasi\b', r'\brekayasa\b', r'\bskenario\b', r'\bdisembunyikan\b',
    r'\bditutupi\b', r'\bdimanipulasi\b', r'\bada yang mengatur\b',
    r'\bada yang bermain\b', r'\bdalang\b', r'\bpersekongkolan\b',
    r'\bsabotase\b', r'\bdidalangi\b', r'\bgerakan bawah tanah\b',
    r'\bsumber terpercaya\b', r'\borang dalam\b', r'\bsumber internal\b',
    r'\btanpa bukti\b', r'\btanpa konfirmasi\b', r'\btanpa verifikasi\b',
    r'\bsanter terdengar\b', r'\bsumber anonim\b', r'\bdisebut-sebut\b',
    r'\bdiisukan\b', r'\bdisinyalir\b', r'\bberedar luas\b', r'\btersiar kabar\b',
    r'\bberita bohong\b', r'\bberita palsu\b', r'\bfake news\b',
    r'\bhoaks\b', r'\bhoax\b', r'\bmenyesatkan\b', r'\bdisesatkan\b',
    r'\bditipu\b', r'\bdibohongi\b', r'\bmanipulasi informasi\b',
    r'\bdistorsi informasi\b', r'\binformasi menyesatkan\b', r'\bpropaganda\b',
    r'\bpencucian otak\b', r'\bnarasi sesat\b', r'\bpenyesatan\b',
    r'\bdiframing\b', r'\bopini sesat\b',
    r'\bpengkhianat\b', r'\btukang bohong\b', r'\btukang tipu\b',
    r'\btukang fitnah\b', r'\bboneka\b', r'\bbadut\b', r'\bpenjilat\b',
    r'\bmunafik\b', r'\bhina dina\b', r'\bnista\b', r'\bterkutuk\b',
    r'\bbejat\b', r'\bbusuk\b',
]

def hitung_skor(judul, isi):
    j, i = str(judul).lower(), str(isi).lower()
    skor = 0
    for pola in KATA_DISTORSI:
        if re.search(pola, j): skor += 2
        elif re.search(pola, i): skor += 1
    return skor

def cek_kata(judul, isi):
    gabung = (str(judul) + " " + str(isi)).lower()
    return [re.search(p, gabung).group() for p in KATA_DISTORSI if re.search(p, gabung)]

df = pd.read_csv(path_gabung, encoding="utf-8-sig")
print(f"\n📂 Total data: {len(df)} artikel")
print("🔍 Proses labeling...")

df["skor_distorsi"] = df.apply(lambda r: hitung_skor(r["judul"], r["isi"]), axis=1)
df["kata_distorsi"] = df.apply(lambda r: ", ".join(cek_kata(r["judul"], r["isi"])), axis=1)
df["label"]         = (df["skor_distorsi"] >= THRESHOLD).astype(int)

n1  = df["label"].sum()
n0  = len(df) - n1
pct = n1 / len(df) * 100

print(f"\n{'='*55}")
print(f"✅ Labeling selesai!  (threshold >= {THRESHOLD})")
print(f"   Label 0 (Normal)   : {n0:>5}  ({100-pct:.1f}%)")
print(f"   Label 1 (Distorsi) : {n1:>5}  ({pct:.1f}%)")
if pct > 80:   print("⚠️  Terlalu banyak distorsi — naikkan THRESHOLD")
elif pct < 25: print("⚠️  Terlalu sedikit distorsi — turunkan THRESHOLD")
else:          print("✅ Distribusi label wajar")

print(f"\n📋 Contoh label 1 (skor tertinggi):")
for _, r in df[df["label"]==1].nlargest(5,"skor_distorsi")[["judul","skor_distorsi","kata_distorsi"]].iterrows():
    print(f"  [{r['skor_distorsi']:>3}] {r['judul'][:65]}")
    print(f"        → {r['kata_distorsi'][:80]}")

df.to_csv(path_labeled, index=False, encoding="utf-8-sig")
print(f"\n💾 {path_labeled}")


# ============================================================
# 3. Balancing
# ============================================================

print("\n⚖️  Balancing dataset...")
min_n = min(len(df[df["label"]==0]), len(df[df["label"]==1]))
df_balanced = pd.concat([
    df[df["label"]==0].sample(n=min_n, random_state=42),
    df[df["label"]==1].sample(n=min_n, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

df_balanced.to_csv(path_balanced, index=False, encoding="utf-8-sig")

print(f"\n{'='*55}")
print(f"✅ Balancing selesai!")
print(f"   Label 0 (Normal)   : {len(df_balanced[df_balanced['label']==0])}")
print(f"   Label 1 (Distorsi) : {len(df_balanced[df_balanced['label']==1])}")
print(f"   Total              : {len(df_balanced)}")
print(f"💾 {path_balanced}")


# ============================================================
# 4. Visualisasi Label
# ============================================================

total    = len(df_balanced)
normal   = len(df_balanced[df_balanced["label"] == 0])
distorsi = len(df_balanced[df_balanced["label"] == 1])

print(f"\n{'='*40}")
print(f"Total artikel     : {total}")
print(f"Label 0 (Normal)  : {normal} ({normal/total*100:.1f}%)")
print(f"Label 1 (Distorsi): {distorsi} ({distorsi/total*100:.1f}%)")
print(f"{'='*40}")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# ── Kiri: bar chart ──────────────────────────────────────────
axes[0].bar(["Normal (0)", "Distorsi (1)"], [normal, distorsi],
            color=["steelblue", "tomato"])
axes[0].set_title("Distribusi Label (Balanced)")
axes[0].set_ylabel("Jumlah Artikel")
for i, v in enumerate([normal, distorsi]):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# ── Kanan: pie chart ─────────────────────────────────────────
axes[1].pie([normal, distorsi],
            labels=["Normal (0)", "Distorsi (1)"],
            colors=["steelblue", "tomato"],
            autopct="%1.1f%%", startangle=90)
axes[1].set_title("Proporsi Label")

plt.suptitle("Distribusi Label Dataset", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "visualisasi_label.png"), dpi=150)
plt.show()

print(f"\n📊 Visualisasi disimpan → {os.path.join(SAVE_DIR, 'visualisasi_label.png')}")